In [2]:
print("hello")

hello


In [3]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.10.0+cu128
True
Tesla T4


In [4]:
import torch

# A tensor is just an n-dimensional array
scalar = torch.tensor(3.14)           # 0D
vector = torch.tensor([1.0, 2.0, 3.0])  # 1D
matrix = torch.tensor([[1.0, 2.0],
                        [3.0, 4.0]])  # 2D
image  = torch.zeros(3, 64, 64)      # 3D — like a real image (C, H, W)

print(scalar.shape)   # torch.Size([])
print(vector.shape)   # torch.Size([3])
print(matrix.shape)   # torch.Size([2, 2])
print(image.shape)    # torch.Size([3, 64, 64])

torch.Size([])
torch.Size([3])
torch.Size([2, 2])
torch.Size([3, 64, 64])


In [5]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print(a + b)        # elementwise add
print(a * b)        # elementwise multiply
print(a.mean())     # mean
print(a.sum())      # sum
print(a @ b)        # dot product (will use this a lot)

# Shape manipulation — critical skill
x = torch.zeros(4, 3)      # 4 rows, 3 cols
print(x.shape)

x = x.reshape(2, 6)        # same data, different shape
print(x.shape)

x = x.unsqueeze(0)         # add a dimension at position 0
print(x.shape)             # (1, 2, 6) — simulates a batch of 1

tensor([5., 7., 9.])
tensor([ 4., 10., 18.])
tensor(2.)
tensor(6.)
tensor(32.)
torch.Size([4, 3])
torch.Size([2, 6])
torch.Size([1, 2, 6])


In [9]:
# By default tensors live on CPU
x = torch.tensor([1.0, 2.0, 3.0])
print(x.device)   # cpu

# Move to GPU
x = x.to('cuda')
print(x.device)   # cuda:0

# Create directly on GPU
y = torch.ones(3, device='cuda')
print(y.device)

# Operations between tensors must be on the same device
# This will CRASH — uncomment to see the error
z = torch.tensor([1.0, 2.0, 3.0])  # cpu
result = x + z                      # cpu + gpu = error

# Move back to CPU
x = x.to('cpu')
print(x.device)

cpu
cuda:0
cuda:0


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

In [10]:
# requires_grad=True tells PyTorch to track operations on this tensor
x = torch.tensor(3.0, requires_grad=True)

# Define a simple function: y = x^2 + 2x + 1
y = x**2 + 2*x + 1

print(f"x = {x}")
print(f"y = {y}")

# Compute gradients — dy/dx
y.backward()

# dy/dx = 2x + 2, at x=3 → 2(3) + 2 = 8
print(f"dy/dx = {x.grad}")  # should be 8.0

x = 3.0
y = 16.0
dy/dx = 8.0


In [11]:
# Pretend this is a model weight we want to optimize
w = torch.tensor(3.0, requires_grad=True)

# Our "loss" — we want to minimize w^2 (minimum is at w=0)
loss = w**2

# Backward pass — compute gradient
loss.backward()

print(f"w = {w.item()}")
print(f"gradient = {w.grad.item()}")   # 2w = 6.0

# Manually update weight (gradient descent)
learning_rate = 0.1
with torch.no_grad():   # don't track this operation
    w -= learning_rate * w.grad

print(f"w after one step = {w.item()}")  # should move closer to 0

w = 3.0
gradient = 6.0
w after one step = 2.4000000953674316


In [12]:
import torch

# True relationship we want the model to discover: y = 3x + 2
# Model doesn't know this — it has to learn it
torch.manual_seed(42)

X = torch.randn(100, 1)           # 100 data points
y = 3 * X + 2 + torch.randn(100, 1) * 0.1  # y = 3x + 2 + noise

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Sample X: {X[:3].squeeze()}")
print(f"Sample y: {y[:3].squeeze()}")

X shape: torch.Size([100, 1])
y shape: torch.Size([100, 1])
Sample X: tensor([1.9269, 1.4873, 0.9007])
Sample y: tensor([7.8534, 6.4710, 4.6632])


In [13]:
# One weight and one bias — simplest possible linear model
# y_pred = w * x + b
torch.manual_seed(42)

w = torch.randn(1, requires_grad=True)   # random init
b = torch.randn(1, requires_grad=True)   # random init

print(f"Initial w: {w.item():.4f}")   # random, far from 3.0
print(f"Initial b: {b.item():.4f}")   # random, far from 2.0

Initial w: 0.3367
Initial b: 0.1288


In [14]:
learning_rate = 0.1
epochs = 100

for epoch in range(epochs):
    # 1. Forward pass — make predictions
    y_pred = w * X + b

    # 2. Loss — Mean Squared Error (how wrong are we?)
    loss = ((y_pred - y) ** 2).mean()

    # 3. Backward pass — compute gradients
    loss.backward()

    # 4. Update weights — gradient descent
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad

    # 5. Zero gradients — CRITICAL, explained below
    w.grad.zero_()
    b.grad.zero_()

    # Print every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1} | Loss: {loss.item():.4f} | w: {w.item():.4f} | b: {b.item():.4f}")

Epoch 10 | Loss: 0.1786 | w: 2.7187 | b: 1.8447
Epoch 20 | Loss: 0.0096 | w: 2.9706 | b: 1.9911
Epoch 30 | Loss: 0.0078 | w: 2.9978 | b: 2.0027
Epoch 40 | Loss: 0.0078 | w: 3.0008 | b: 2.0035
Epoch 50 | Loss: 0.0078 | w: 3.0011 | b: 2.0036
Epoch 60 | Loss: 0.0078 | w: 3.0012 | b: 2.0036
Epoch 70 | Loss: 0.0078 | w: 3.0012 | b: 2.0036
Epoch 80 | Loss: 0.0078 | w: 3.0012 | b: 2.0036
Epoch 90 | Loss: 0.0078 | w: 3.0012 | b: 2.0036
Epoch 100 | Loss: 0.0078 | w: 3.0012 | b: 2.0036


In [15]:
# Reset weights
torch.manual_seed(42)
w = torch.randn(1, requires_grad=True)
b = torch.randn(1, requires_grad=True)

# Adam optimizer — handles the update step for you
optimizer = torch.optim.Adam([w, b], lr=0.1)

for epoch in range(100):
    # Forward pass
    y_pred = w * X + b

    # Loss
    loss = ((y_pred - y) ** 2).mean()

    # Backward
    loss.backward()

    # Optimizer handles the weight update
    optimizer.step()

    # Zero gradients — still your job
    optimizer.zero_grad()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1} | Loss: {loss.item():.4f} | w: {w.item():.4f} | b: {b.item():.4f}")

Epoch 10 | Loss: 4.2581 | w: 1.3197 | b: 1.1027
Epoch 20 | Loss: 0.8207 | w: 2.1913 | b: 1.8902
Epoch 30 | Loss: 0.1162 | w: 2.8218 | b: 2.2758
Epoch 40 | Loss: 0.0931 | w: 3.1359 | b: 2.2501
Epoch 50 | Loss: 0.0487 | w: 3.1854 | b: 2.0573
Epoch 60 | Loss: 0.0236 | w: 3.1062 | b: 1.9350
Epoch 70 | Loss: 0.0125 | w: 3.0193 | b: 1.9426
Epoch 80 | Loss: 0.0083 | w: 2.9782 | b: 1.9992
Epoch 90 | Loss: 0.0088 | w: 2.9779 | b: 2.0252
Epoch 100 | Loss: 0.0080 | w: 2.9928 | b: 2.0143


In [1]:
import torch
import torch.nn as nn

# Simplest possible nn.Module
class LinearModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)  # 1 input, 1 output

    def forward(self, x):
        return self.linear(x)

# Create the model
model = LinearModel()
print(model)
print(f"Parameters: {list(model.parameters())}")

LinearModel(
  (linear): Linear(in_features=1, out_features=1, bias=True)
)
Parameters: [Parameter containing:
tensor([[-0.7849]], requires_grad=True), Parameter containing:
tensor([-0.8601], requires_grad=True)]


In [2]:
# Same data from Day 2
torch.manual_seed(42)
X = torch.randn(100, 1)
y = 3 * X + 2 + torch.randn(100, 1) * 0.1

# Model, loss function, optimizer
model = LinearModel()
criterion = nn.MSELoss()                           # replaces ((y_pred - y)**2).mean()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)  # passes all params automatically

for epoch in range(100):
    # Forward pass
    y_pred = model(X)                  # calls forward() automatically

    # Loss
    loss = criterion(y_pred, y)

    # Backward
    loss.backward()

    # Update + zero grads
    optimizer.step()
    optimizer.zero_grad()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1} | Loss: {loss.item():.4f}")

# Check what the model learned
print(f"\nLearned w: {model.linear.weight.item():.4f}")
print(f"Learned b: {model.linear.bias.item():.4f}")

Epoch 10 | Loss: 0.1400
Epoch 20 | Loss: 0.0093
Epoch 30 | Loss: 0.0078
Epoch 40 | Loss: 0.0078
Epoch 50 | Loss: 0.0078
Epoch 60 | Loss: 0.0078
Epoch 70 | Loss: 0.0078
Epoch 80 | Loss: 0.0078
Epoch 90 | Loss: 0.0078
Epoch 100 | Loss: 0.0078

Learned w: 3.0012
Learned b: 2.0036


In [3]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(10, 64),   # input layer  — 10 features in, 64 out
            nn.ReLU(),           # activation — adds non-linearity
            nn.Linear(64, 32),  # hidden layer — 64 in, 32 out
            nn.ReLU(),
            nn.Linear(32, 1)    # output layer — 32 in, 1 out
        )

    def forward(self, x):
        return self.network(x)

model = MLP()
print(model)

# Count total parameters
total = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total:,}")

# Pass dummy data through
x = torch.randn(8, 10)     # batch of 8 samples, 10 features each
out = model(x)
print(f"\nInput shape:  {x.shape}")
print(f"Output shape: {out.shape}")

MLP(
  (network): Sequential(
    (0): Linear(in_features=10, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)

Total parameters: 2,817

Input shape:  torch.Size([8, 10])
Output shape: torch.Size([8, 1])
